This notebook is dedicated to Data Formation (and all the .cvs files).

# Image Features
Extracting features from all the images. Features include:
- hue(color type)
- saturation(color intensity)
- value(brightness)
- contrast :std(img_gray)
- colorfulness : a formula which R, G, and B are used

### This part icludes four stages:
1. features determination
2. extracting determined features from all the images and saved as "image_features"
3. aggregating; took a mean of the images' features related to same story and saved as "story_features.csv"
4. standardization; saved as "story_features_standardized.csv"

In [ ]:
#----------------------------------------------------
# Path
#----------------------------------------------------

# ctrl + / --> comment

import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler

image_folder = "C:/sepide/00-UTwente/Courses and Thesis/Final Project/data/Coding/images"

os.makedirs("data", exist_ok=True)

data_path = "../data"
image_features_path = "data/image_features.csv"
story_features_path = "data/story_features_v2.csv"
standardized_story_features_path = "data/story_features_standardized_v2.csv"

valid_extensions = (".png",)

In [ ]:
#-------------------------------------
# Features from the images
#-------------------------------------

def extract_image_features(image_path):
    # Read image
    img_bgr = cv2.imread(image_path)
        # To check whether all the images are readable
    if img_bgr is None:
        raise ValueError(f"Could not read image: {image_path}")

    # Resize for consistency (??????????????????????????????????????????????????????????/)
    img_bgr = cv2.resize(img_bgr, (256, 256))

    # Convert formats
    # RGB: red, green, blue
    # HSV: hue(color type), saturation(color intensity), value(brightness/lightness)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # Split channels
    r, g, b = cv2.split(img_rgb)
    h, s, v = cv2.split(img_hsv)

    # Basic stats
    # Including both mean and std (dispersion of each feature)
    features = {
        #d "mean_red": float(np.mean(r)),
        #d "std_red": float(np.std(r)),
        #d "mean_green": float(np.mean(g)),
        #d "std_green": float(np.std(g)),
        #d "mean_blue": float(np.mean(b)),
        #d "std_blue": float(np.std(b)),

        "mean_hue": float(np.mean(h)), #range: [0, 179]
        "std_hue": float(np.std(h)),
        "mean_saturation": float(np.mean(s)),   #range: [0, 255] # 0-->more muted and grayish colors, 1-->vivid and strong colors
        "std_saturation": float(np.std(s)),
        "mean_brightness": float(np.mean(v)),   #range: [0, 255] # 0--> dark image, 1--> bright image
        "std_brightness": float(np.std(v)),

        #d "gray_mean": float(np.mean(img_gray)),
        #d "gray_std": float(np.std(img_gray)),   # useful as contrast proxy
        # "entropy": float(shannon_entropy(img_gray)),
    }

    # Contrast (another useful proxy)
    features["contrast"] = float(img_gray.std())

    # # Edge density
    # edges = cv2.Canny(img_gray, threshold1=100, threshold2=200)
    # features["edge_density"] = float(np.mean(edges > 0))

    # Colorfulness (Hasler & Süsstrunk style - 2003)
    # f as float
    r_f = r.astype(np.float32)
    g_f = g.astype(np.float32)
    b_f = b.astype(np.float32)

    rg = np.abs(r_f - g_f)
    yb = np.abs(0.5 * (r_f + g_f) - b_f)

    std_rg, mean_rg = np.std(rg), np.mean(rg)
    std_yb, mean_yb = np.std(yb), np.mean(yb)

    colorfulness = np.sqrt(std_rg**2 + std_yb**2) + 0.3 * np.sqrt(mean_rg**2 + mean_yb**2)
    features["colorfulness"] = float(colorfulness)

    return features

In [ ]:
#-------------------------------------
# Extracting features from all the images
#-------------------------------------

rows = []

for filename in tqdm(sorted(os.listdir(image_folder))):
    if filename.lower().endswith(valid_extensions):
        image_path = os.path.join(image_folder, filename)

        try:
            features = extract_image_features(image_path)
            features["image_file"] = filename

            # optional: extract story_id from filename like HT_27_1.png
            name, ext = os.path.splitext(filename)
            parts = name.split("_")
            if len(parts) >= 2:
                features["story_id"] = f"{parts[0]}_{parts[1]}"
            else:
                features["story_id"] = None

            rows.append(features)

        except Exception as e:
            print(f"Error processing {filename}: {e}")

# Convert to DataFrame
df_images = pd.DataFrame(rows)
# To add "story_id" as first column
cols = ["story_id"] + [col for col in df_images.columns if col != "story_id"]
df_images = df_images[cols]

# Save to CSV
os.makedirs("data", exist_ok=True)
df_images.to_csv(f"{data_path}/image_features.csv", index=False)

print(df_images.head())
print(f"\nTotal processed images: {len(df_images)}")

In [ ]:
#-------------------------------------
# Aggregating image features
#-------------------------------------

df_images = pd.read_csv(image_features_path)
#Columns from image_features data: story_id,mean_hue,std_hue,mean_saturation,std_saturation,mean_brightness,std_brightness,contrast,colorfulness,image_file
story_features = (
    df_images
    .groupby("story_id")
    .agg({
        "mean_hue": ["mean"],
        "mean_saturation": ["mean"],
        "mean_brightness": ["mean"],
        "contrast": ["mean"],
        "colorfulness": ["mean"],
        "image_file": "count"
    })
)

story_features.columns = [
    "hue_mean",
    "saturation_mean",
    "brightness_mean",
    "contrast_mean",
    "colorfulness_mean",
    "num_images"
]

story_features = story_features.reset_index()

story_features.to_csv(f"{data_path}/story_features_v2.csv", index=False)

print(story_features.head())

In [ ]:
#-------------------------------------
# Standardization of story features
#-------------------------------------

story_features = pd.read_csv(story_features_path)

feature_cols = [
    "hue_mean",
    "saturation_mean",
    "brightness_mean",
    "contrast_mean",
    "colorfulness_mean",
    "num_images"
]

# Save actual values
story_features["num_images_actual"] = story_features["num_images"]

# Standardize
scaler = StandardScaler()
story_features[feature_cols] = scaler.fit_transform(story_features[feature_cols])

# Move actual number of images column to the end
cols = [col for col in story_features.columns if col != "num_images_actual"] + ["num_images_actual"]
# Re-ordering columns based on cols order in above
story_features = story_features[cols]

# Save
story_features.to_csv(f"{data_path}/story_features_standardized_v2.csv", index=False)

print(story_features.head())
print(story_features.shape)


# Student Preference and Images Features
Merging student preference and extracted images' features per story.

How data looks like?
Columns: story id, student id, preference, feature 1, ..., feature x
* preference is a variable that has different value in each row and do not repeated.

In [ ]:
#-----------------------------
# Students' preference cleaning
#-----------------------------
import pandas as pd

# Paths
raw_pref_path = "../data/student_pref_raw.csv"
clean_pref_path = "../data/student_pref_clean.csv"

# Read raw student preference data
df_raw = pd.read_csv(raw_pref_path)

# Add student_id
df_raw["student_id"] = range(1, len(df_raw) + 1)

# Convert wide to long format
df_long = df_raw.melt(
    id_vars="student_id",
    var_name="story_id",
    value_name="preference"
)

# Remove non-read stories
df_long = df_long[df_long["preference"] != 0]

# Clean story_id
df_long["story_id"] = df_long["story_id"].str.extract(r"(HT_\d+)")

# Save cleaned data
df_long.to_csv(clean_pref_path, index=False)

print(df_long.head())
print(df_long.shape)

In [5]:
#------------------------------------
# Merging data based on story_id
#------------------------------------

import pandas as pd

# Paths
clean_pref_path = "../data/student_pref_clean.csv"
story_features_path = "../data/story_features_standardized_v2.csv"

# Read saved files
df_students = pd.read_csv(clean_pref_path)
df_story = pd.read_csv(story_features_path)

# Merge based on story_id
df_merged = df_students.merge(
    df_story,
    on="story_id",
    how="inner"
)

df_merged.to_csv("../data/merged_data_v2.csv", index=False)

print(df_merged.head())
print(df_merged.shape)

   student_id story_id  preference  hue_mean  saturation_mean  \
0           1    HT_41           3  0.972431         0.969905   
1          61    HT_41           2  0.972431         0.969905   
2          78    HT_41           1  0.972431         0.969905   
3          93    HT_41           1  0.972431         0.969905   
4          97    HT_41           2  0.972431         0.969905   

   brightness_mean  contrast_mean  colorfulness_mean  num_images  \
0        -0.090928      -0.588275           1.046952    3.955032   
1        -0.090928      -0.588275           1.046952    3.955032   
2        -0.090928      -0.588275           1.046952    3.955032   
3        -0.090928      -0.588275           1.046952    3.955032   
4        -0.090928      -0.588275           1.046952    3.955032   

   num_images_actual  
0                  7  
1                  7  
2                  7  
3                  7  
4                  7  
(14928, 10)


In [3]:
#------------------------------------
# Merging story features and preferences
#------------------------------------

import pandas as pd

# Paths
story_features_path = "../data/story_features_standardized_v2.csv"
student_pref_path = "../data/student_pref_clean.csv"

# Read data
story_features = pd.read_csv(story_features_path)
student_pref = pd.read_csv(student_pref_path)

# Calculate mean preference for each story
story_pref = (
    student_pref
    .groupby("story_id")["preference"]
    .mean()
    .reset_index()
)

# Merge story features with mean preference
story_features_and_pref = story_features.merge(
    story_pref,
    on="story_id",
    # using "inner" to only choose stories that exist in both data set to avoid NaN data
    how="inner"
)

# Save new file
story_features_and_pref.to_csv("../data/story_features_and_pref_v2.csv", index=False)

# Check result
print(story_features_and_pref.head())
print(story_features_and_pref.shape)

  story_id  hue_mean  saturation_mean  brightness_mean  contrast_mean  \
0    HT_01 -1.618578        -1.881846         1.359822      -1.443210   
1    HT_02 -1.229657        -1.290028         1.704306      -0.988076   
2    HT_03  0.011002        -1.297485         1.653669      -0.199927   
3    HT_04  0.187083        -1.407629         1.202232       0.477212   
4    HT_05 -1.338483        -1.800431         1.581200       0.642966   

   colorfulness_mean  num_images  num_images_actual  preference  
0          -1.084601   -0.408370                  2    2.698795  
1          -0.535425    1.336991                  4    3.522013  
2           0.291979   -1.281050                  1    2.838710  
3          -1.051149    2.209671                  5    3.036036  
4          -1.137558    0.464311                  3    3.387755  
(154, 9)
